In [9]:
from pyspark import SparkContext
sc=SparkContext.getOrCreate()
data=[
      ("anita","female","cs","fail"),
      ("priya","female","it","pass"),
      ("Rahul","male","cs","fail"),
      ("neha","female","it","pass"),
      ("vikash","male","ece","pass")
  ]
rdd=sc.parallelize(data)
print(rdd.collect())

[('anita', 'female', 'cs', 'fail'), ('priya', 'female', 'it', 'pass'), ('Rahul', 'male', 'cs', 'fail'), ('neha', 'female', 'it', 'pass'), ('vikash', 'male', 'ece', 'pass')]


In [4]:
# Find all male student

In [11]:
result=rdd.filter(lambda x: x[1] == "male")

print(result.collect())

[('Rahul', 'male', 'cs', 'fail'), ('vikash', 'male', 'ece', 'pass')]


In [12]:
# FInd all female student

In [13]:
result=rdd.filter(lambda x: x[1] == "female")

print(result.collect())

[('anita', 'female', 'cs', 'fail'), ('priya', 'female', 'it', 'pass'), ('neha', 'female', 'it', 'pass')]


In [14]:
# find the cs student

In [15]:
result=rdd.filter(lambda x: x[2] == "cs")

print(result.collect())

[('anita', 'female', 'cs', 'fail'), ('Rahul', 'male', 'cs', 'fail')]


In [16]:
# find the student who passed the exam

In [18]:
result=rdd.filter(lambda x: x[3] == "pass")

print(result.collect())

[('priya', 'female', 'it', 'pass'), ('neha', 'female', 'it', 'pass'), ('vikash', 'male', 'ece', 'pass')]


In [19]:
# Count male and female student

In [28]:
male = rdd.filter(lambda x: x[1] == "male").count()
female = rdd.filter(lambda x: x[1] == "female").count()

print("Male:", male)
print("Female:", female)

Male: 2
Female: 3


In [27]:
print(rdd.take(5))

[('anita', 'female', 'cs', 'fail'), ('priya', 'female', 'it', 'pass'), ('Rahul', 'male', 'cs', 'fail'), ('neha', 'female', 'it', 'pass'), ('vikash', 'male', 'ece', 'pass')]


In [22]:
# each department count

In [23]:
result = rdd.map(lambda x:x[2]).countByValue()
print(result)

defaultdict(<class 'int'>, {'cs': 2, 'it': 2, 'ece': 1})


In [26]:
# Count the no. of pass and fail student

In [30]:
passed = rdd.filter(lambda x: x[3] == "pass").count()
failed = rdd.filter(lambda x: x[3] == "fail").count()

print("Pass:", passed)
print("Fail:", failed)

Pass: 3
Fail: 2


In [29]:
# Find the Female student who passed

In [33]:
fem = rdd.filter(lambda x: x[1] == "female" and x[3]=="pass")
print(fem.collect())

[('priya', 'female', 'it', 'pass'), ('neha', 'female', 'it', 'pass')]


In [32]:
# find out the mail who failed the exam

In [34]:
fem = rdd.filter(lambda x: x[1] == "male" and x[3]=="fail")
print(fem.collect())

[('Rahul', 'male', 'cs', 'fail')]


In [35]:
# get only the student name

In [37]:
names = rdd.map(lambda x: x[0]).collect()

print(names)

['anita', 'priya', 'Rahul', 'neha', 'vikash']


In [38]:
# count student in each result

In [40]:
result=rdd.map(lambda x: (x[3], 1)).reduceByKey(lambda a,b:a+b)
print(result.collect())

[('fail', 2), ('pass', 3)]


In [41]:
# count the student in each department

In [42]:
result=rdd.map(lambda x: (x[2], 1)).reduceByKey(lambda a,b:a+b)
print(result.collect())

[('it', 2), ('ece', 1), ('cs', 2)]


In [43]:
# add three no. based on lambda

In [44]:
add = lambda x, y, z: x + y + z

print(add(10, 20, 30))

60


In [45]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('ml-bank').getOrCreate()
df= spark.read.csv('/content/bank.csv',header=True,inferSchema=True)
df.printSchema()

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- deposit: string (nullable = true)



In [47]:
numeric_feature =[t[0] for t in df.dtypes if t[1] == 'int']
df.select(numeric_feature).describe().toPandas()

,summary,age,balance,day,duration,campaign,pdays,previous
0,count,11162,11162,11162,11162,11162,11162,11162
1,mean,41.231947679627304,1528.5385235620856,15.658036194230425,371.99381831213043,2.508421429851281,51.33040673714388,0.8325568894463358
2,stddev,11.913369192215518,3225.413325946149,8.420739541006462,347.12838571630687,2.7220771816614824,108.75828197197717,2.292007218670508
3,min,18,-6847,1,2,1,-1,0
4,max,95,81204,31,3881,63,854,58


In [52]:
from pyspark.ml.feature import StringIndexer,OneHotEncoder, VectorAssembler

# Categorical columns
categoricalColumns = [
    'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'poutcome'
]

stages = []

# Convert categorical values to numbers and then to vectors
for col in categoricalColumns:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index")
    encoder = OneHotEncoder(
        inputCol=col + "_Index",
        outputCol=col + "_Vec"
    )
    stages.append(indexer)
    stages.append(encoder)

# Convert target column into label
labelIndexer = StringIndexer(inputCol="deposit", outputCol="label")
stages.append(labelIndexer)

# Numerical columns
numericCols = [
    'age', 'balance', 'duration',
    'campaign', 'pdays', 'previous'
]

# Combine all columns into one features column
featureCols = [col + "_Vec" for col in categoricalColumns] + numericCols

assembler = VectorAssembler(
    inputCols=featureCols,
    outputCol="features"
)

stages.append(assembler)

In [53]:
stages

[StringIndexer_c4569a7f1a45,
 OneHotEncoder_a0e9831a26a8,
 StringIndexer_d8c2170d6ebf,
 OneHotEncoder_2c6d221a1ee7,
 StringIndexer_1547cf02f4f7,
 OneHotEncoder_686df9ac08e6,
 StringIndexer_99d3674ac5b0,
 OneHotEncoder_9e5e32df7880,
 StringIndexer_ca873855bf8a,
 OneHotEncoder_ce15d62d1280,
 StringIndexer_fd57f825609a,
 OneHotEncoder_159add4a0241,
 StringIndexer_88d416db1023,
 OneHotEncoder_8c70308b34a3,
 StringIndexer_74a83961daba,
 OneHotEncoder_123a7129efac,
 StringIndexer_05e74555c8c9,
 VectorAssembler_5e80b8421fbb]

In [54]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages = stages)
pipelineModel = pipeline.fit(df)
df = pipelineModel.transform(df)
